## Search evaluation with ground truth

In [2]:
# For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

# Load the ground truth file from the previous notebook:

import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

# Use the same ingest.py file we downloaded in the previous notebook.

# Load the documents and build a minsearch index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)



In [4]:
# Wrap the search call in a function called text_search. The name is deliberate. 
# Later we'll write vector_search or a hybrid version and run the exact same evaluation on it. 
# Everything downstream only needs a function that takes a query and returns results, so we can swap one for another. 
# That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [5]:
# Collecting relevance data
# Start with one ground truth record:

q = ground_truth[0]
q

# Run search for this question:

doc_id = q["document"]
results = text_search(query=q["question"])

# First, compare the retrieved document IDs with the correct document ID:

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [6]:
# Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is 
# the correct document for this question.

relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

# This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document.

# Put this logic into a function:

def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

    

In [7]:
# For the first ground truth record, the relevance list is:

q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

# The correct document was the first search result.

Can I still join the course if I just found it now?


[1, 0, 0, 0, 0]

In [8]:
# For this question:

q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

# Here, the correct document was also the first search result.

Where is the live stream link for the workshop or Office Hours usually announced before it starts?


[1, 0, 0, 0, 0]

In [9]:
# For this question:

q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

# The correct document was found at the first position again.

Where do I find the LLM Zoomcamp course page with the syllabus and deadlines?


[1, 0, 0, 0, 0]

In [10]:
# Now do the same thing for all ground truth questions:

from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total
    
# Call it for the first 15 ground truth questions:

ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)


  0%|          | 0/15 [00:00<?, ?it/s]

In [11]:
# Look at the results:

relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [12]:
# Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, 
# hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance
    
# The total relevance function gets a search_function too.

In [13]:
# We need to provide it explicitly:

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total
    
# Use it with text_search on the same sample:

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [14]:
# Now run it for all ground truth questions:

relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

## Search Evaluation Metrics